In [0]:
"""
01_generate_synthetic_data

Synthetic healthcare data generator for a readmission risk use case.

This notebook:
- Creates synthetic patients, encounters, diagnoses, medications, and lab results.
- Writes them as Delta tables in a chosen catalog/schema (or the default catalog).

You can run this as a Databricks notebook (import this .py file) or as a script.
"""


In [0]:
from datetime import datetime, timedelta
import random

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType,
    BooleanType,
)

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

CATALOG_NAME = 'healthcare'  
SCHEMA_NAME = "readmission"

N_PATIENTS = 1000
MAX_ENCOUNTERS_PER_PATIENT = 10
SEED = 42


In [0]:
def get_spark() -> SparkSession:
    return SparkSession.builder.getOrCreate()

def qualify_table(table_name: str) -> str:
    """
    Build a fully-qualified table name if catalog/schema are provided.
    """
    if CATALOG_NAME and SCHEMA_NAME:
        return f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    if SCHEMA_NAME:
        return f"{SCHEMA_NAME}.{table_name}"
    return table_name


In [0]:
def create_base_dates():
    today = datetime.today().date()
    start_date = today - timedelta(days=365 * 2)
    return start_date, today


In [0]:
def generate_patients(spark: SparkSession):
    random.seed(SEED)
    start_date, _ = create_base_dates()

    genders = ["M", "F"]
    smoking_statuses = ["never", "former", "current"]

    rows = []
    for pid in range(1, N_PATIENTS + 1):
        age = random.randint(18, 90)
        gender = random.choice(genders)
        sm = random.choice(smoking_statuses)
        zipcode = f"{random.randint(10000, 99999)}"
        registration_date = start_date + timedelta(days=random.randint(0, 365))

        rows.append(
            (
                f"P{pid:06d}",
                age,
                gender,
                sm,
                zipcode,
                registration_date,
            )
        )

    schema = StructType(
        [
            StructField("patient_id", StringType(), False),
            StructField("age", IntegerType(), True),
            StructField("gender", StringType(), True),
            StructField("smoking_status", StringType(), True),
            StructField("zipcode", StringType(), True),
            StructField("registration_date", DateType(), True),
        ]
    )

    return spark.createDataFrame(rows, schema=schema)

In [0]:
def generate_encounters(spark: SparkSession):
    random.seed(SEED + 1)
    start_date, end_date = create_base_dates()

    encounter_types = ["inpatient", "outpatient", "emergency"]

    rows = []
    encounter_id_counter = 1
    for pid in range(1, N_PATIENTS + 1):
        patient_id = f"P{pid:06d}"
        n_encounters = random.randint(1, MAX_ENCOUNTERS_PER_PATIENT)

        encounter_dates = sorted(
            [
                start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
                for _ in range(n_encounters)
            ]
        )

        for ed in encounter_dates:
            length_of_stay = random.randint(1, 15)
            discharge_date = ed + timedelta(days=length_of_stay)
            etype = random.choice(encounter_types)

            readmit_flag = random.random() < 0.2
            rows.append(
                (
                    f"E{encounter_id_counter:08d}",
                    patient_id,
                    etype,
                    ed,
                    discharge_date,
                    length_of_stay,
                    readmit_flag,
                )
            )
            encounter_id_counter += 1

    schema = StructType(
        [
            StructField("encounter_id", StringType(), False),
            StructField("patient_id", StringType(), False),
            StructField("encounter_type", StringType(), True),
            StructField("admit_date", DateType(), True),
            StructField("discharge_date", DateType(), True),
            StructField("length_of_stay", IntegerType(), True),
            StructField("readmitted_30d", BooleanType(), True),
        ]
    )

    return spark.createDataFrame(rows, schema=schema)

In [0]:
def generate_diagnoses(spark: SparkSession, encounters_df):
    random.seed(SEED + 2)

    diagnosis_codes = [
        "I10",
        "E11.9",
        "J45.909",
        "I50.9",
        "N18.9",
        "E78.5",
        "F17.200",
        "E66.9",
        "K21.9",
        "M54.5",
    ]

    encounter_ids = [row["encounter_id"] for row in encounters_df.select("encounter_id").collect()]

    rows = []
    for eid in encounter_ids:
        n_diag = random.randint(1, 4)
        codes = random.sample(diagnosis_codes, n_diag)
        for idx, code in enumerate(codes, start=1):
            rows.append(
                (
                    eid,
                    code,
                    idx,
                )
            )

    schema = StructType(
        [
            StructField("encounter_id", StringType(), False),
            StructField("diagnosis_code", StringType(), False),
            StructField("diagnosis_rank", IntegerType(), True),
        ]
    )

    return spark.createDataFrame(rows, schema=schema)

In [0]:
def generate_medications(spark: SparkSession, encounters_df):
    random.seed(SEED + 3)

    meds = [
        "metformin",
        "lisinopril",
        "atorvastatin",
        "albuterol",
        "furosemide",
        "insulin",
        "omeprazole",
        "aspirin",
    ]

    encounter_ids = [row["encounter_id"] for row in encounters_df.select("encounter_id").collect()]

    rows = []
    for eid in encounter_ids:
        n_meds = random.randint(0, 3)
        if n_meds == 0:
            continue
        selected_meds = random.sample(meds, n_meds)
        for m in selected_meds:
            rows.append((eid, m))

    schema = StructType(
        [
            StructField("encounter_id", StringType(), False),
            StructField("medication_name", StringType(), False),
        ]
    )

    return spark.createDataFrame(rows, schema=schema)

In [0]:
def generate_labs(spark: SparkSession, encounters_df):
    random.seed(SEED + 4)

    lab_tests = ["glucose", "creatinine", "hemoglobin", "wbc", "platelets"]

    encounter_ids = [row["encounter_id"] for row in encounters_df.select("encounter_id").collect()]

    rows = []
    for eid in encounter_ids:
        for lab in lab_tests:
            value = None
            if lab == "glucose":
                value = random.gauss(120, 30)
            elif lab == "creatinine":
                value = random.gauss(1.0, 0.3)
            elif lab == "hemoglobin":
                value = random.gauss(13.5, 2)
            elif lab == "wbc":
                value = random.gauss(7.0, 2)
            elif lab == "platelets":
                value = random.gauss(250, 60)

            rows.append((eid, lab, float(max(value, 0.0))))

    schema = StructType(
        [
            StructField("encounter_id", StringType(), False),
            StructField("lab_name", StringType(), False),
            StructField("lab_value", DoubleType(), True),
        ]
    )

    return spark.createDataFrame(rows, schema=schema)

In [0]:
def main():
    spark = get_spark()

    if CATALOG_NAME and SCHEMA_NAME:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS healthcare")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS healthcare.readmission")
        spark.sql(f"USE CATALOG healthcare")
        spark.sql(f"USE SCHEMA readmission")
    elif SCHEMA_NAME:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS readmission")
        spark.sql(f"USE SCHEMA readmission")

    patients_df = generate_patients(spark)
    encounters_df = generate_encounters(spark)
    diagnoses_df = generate_diagnoses(spark, encounters_df)
    medications_df = generate_medications(spark, encounters_df)
    labs_df = generate_labs(spark, encounters_df)

    patients_df.write.mode("overwrite").format("delta").saveAsTable(qualify_table("bronze_patients"))
    encounters_df.write.mode("overwrite").format("delta").saveAsTable(qualify_table("bronze_encounters"))
    diagnoses_df.write.mode("overwrite").format("delta").saveAsTable(qualify_table("bronze_diagnoses"))
    medications_df.write.mode("overwrite").format("delta").saveAsTable(qualify_table("bronze_medications"))
    labs_df.write.mode("overwrite").format("delta").saveAsTable(qualify_table("bronze_labs"))

    print("Synthetic bronze tables created:")
    for t in [
        "bronze_patients",
        "bronze_encounters",
        "bronze_diagnoses",
        "bronze_medications",
        "bronze_labs",
    ]:
        print(f"- {qualify_table(t)}")


if __name__ == "__main__":
    main()

In [0]:
%sql
select * from bronze_patients

In [0]:
%sql
select * from bronze_encounters

In [0]:
%sql
select * from bronze_diagnoses

In [0]:
%sql
select * from bronze_medications

In [0]:
%sql
select * from bronze_labs